In [ ]:
!pip install -q scipy pywavelets scikit-learn
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, cohen_kappa_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
import scipy.io as sio
from scipy import signal
from scipy.ndimage import zoom
import numpy as np
import os
import copy
import urllib.request
import urllib.error
import random
import csv
import pywt
import gc

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
DATASET_INFO = {
    'IndianPines': {
        'data_url': 'http://www.ehu.eus/ccwintco/uploads/6/67/Indian_pines_corrected.mat',
        'gt_url':   'http://www.ehu.eus/ccwintco/uploads/c/c4/Indian_pines_gt.mat',
        'data_key': 'indian_pines_corrected',
        'gt_key':   'indian_pines_gt'
    },
    'PaviaU': {
        'data_url': 'http://www.ehu.eus/ccwintco/uploads/e/ee/PaviaU.mat',
        'gt_url':   'http://www.ehu.eus/ccwintco/uploads/5/50/PaviaU_gt.mat',
        'data_key': 'paviaU',
        'gt_key':   'paviaU_gt'
    },
    'Salinas': {
        'data_url': 'http://www.ehu.eus/ccwintco/uploads/a/a3/Salinas_corrected.mat',
        'gt_url':   'http://www.ehu.eus/ccwintco/uploads/f/fa/Salinas_gt.mat',
        'data_key': 'salinas_corrected',
        'gt_key':   'salinas_gt'
    },
    'Botswana': {
        'data_url': 'http://www.ehu.eus/ccwintco/uploads/7/72/Botswana.mat',
        'gt_url':   'http://www.ehu.eus/ccwintco/uploads/5/58/Botswana_gt.mat',
        'data_key': 'Botswana',
        'gt_key':   'Botswana_gt'
    },
    'KSC': {
        'data_url': 'http://www.ehu.eus/ccwintco/uploads/2/26/KSC.mat',
        'gt_url':   'http://www.ehu.eus/ccwintco/uploads/a/a6/KSC_gt.mat',
        'data_key': 'KSC',
        'gt_key':   'KSC_gt'
    },
    'WHU_Hi_HanChuan': {
        'data_url': '',
        'gt_url':   '',
        'data_key': 'WHU_Hi_HanChuan',
        'gt_key':   'WHU_Hi_HanChuan_gt'
    }
}

def load_dataset(dataset_name, data_dir='/kaggle/input/datasets/devgurucodes/datasets-hsi-all'):
    if dataset_name not in DATASET_INFO:
        raise ValueError(f"Dataset {dataset_name} not found in DATASET_INFO.")
        
    info = DATASET_INFO[dataset_name]
    
    data_path = os.path.join(data_dir, f"{dataset_name}_data.mat")
    gt_path = os.path.join(data_dir, f"{dataset_name}_gt.mat")
    
    if not os.path.exists(data_path):
        print(f"Warning: {data_path} not found. Attempting alternative Kaggle path...")
        data_dir = '/kaggle/input/datasets-hsi-all'
        data_path = os.path.join(data_dir, f"{dataset_name}_data.mat")
        gt_path = os.path.join(data_dir, f"{dataset_name}_gt.mat")
            
    img = sio.loadmat(data_path)[info['data_key']]
    gt = sio.loadmat(gt_path)[info['gt_key']]
    return img, gt


In [ ]:
def apply_pca(X, num_components=30, whiten=False):
    newX = np.reshape(X, (-1, X.shape[2]))
    pca = PCA(n_components=num_components, whiten=whiten)
    newX = pca.fit_transform(newX)
    newX = np.reshape(newX, (X.shape[0], X.shape[1], num_components))
    return newX.astype(np.float32)

def pad_with_zeros(X, margin=2):
    newX = np.zeros((X.shape[0] + 2 * margin, X.shape[1] + 2* margin, X.shape[2]), dtype=np.float32)
    x_offset = margin
    y_offset = margin
    newX[x_offset:X.shape[0] + x_offset, y_offset:X.shape[1] + y_offset, :] = X
    return newX

def apply_kalman_denoising(X):
    print("  Applying AFDKF denoising...")
    denoised = np.zeros_like(X, dtype=np.float32)
    for b in range(X.shape[2]):
        band = X[:, :, b].astype(np.float64)
        denoised[:, :, b] = signal.wiener(band, mysize=(5, 5)).astype(np.float32)
    return denoised

def apply_polar_wavelet_transform(X):
    print("  Applying PLCWT feature extraction...")
    h, w, b = X.shape
    ll_features = np.zeros((h, w, b), dtype=np.float32)
    for band_idx in range(b):
        band = X[:, :, band_idx].astype(np.float64)
        coeffs = pywt.dwt2(band, 'db4')
        LL, (LH, HL, HH) = coeffs
        zoom_h = h / LL.shape[0]
        zoom_w = w / LL.shape[1]
        ll_resized = zoom(LL, (zoom_h, zoom_w), order=1).astype(np.float32)
        ll_features[:, :, band_idx] = ll_resized[:h, :w]
    combined = np.concatenate([X, ll_features], axis=2)
    print(f"  PLCWT: {b} bands -> {combined.shape[2]} bands")
    return combined

def create_disjoint_patches(X, y, window_size=11, train_ratio=0.05, seed=42):
    print(f"  Extracting disjoint patches (Train: {train_ratio*100}%)...")
    rng = np.random.RandomState(seed)
    margin = (window_size - 1) // 2
    padded_X = pad_with_zeros(X, margin)

    X_train_list, y_train_list, X_test_list, y_test_list = [], [], [], []
    classes = np.unique(y[y > 0])

    for c in classes:
        class_indices = np.argwhere(y == c)
        rng.shuffle(class_indices)
        n_train = max(1, int(len(class_indices) * train_ratio))
        train_centers = class_indices[:n_train]
        test_centers  = class_indices[n_train:]

        for r, c_idx in train_centers:
            patch = padded_X[r:r+window_size, c_idx:c_idx+window_size, :]
            patch = np.transpose(patch, (2, 0, 1))
            X_train_list.append(patch)
            y_train_list.append(int(c - 1))

        for r, c_idx in test_centers:
            patch = padded_X[r:r+window_size, c_idx:c_idx+window_size, :]
            patch = np.transpose(patch, (2, 0, 1))
            X_test_list.append(patch)
            y_test_list.append(int(c - 1))

    return np.array(X_train_list, dtype=np.float32), np.array(X_test_list, dtype=np.float32), np.array(y_train_list, dtype=np.int64), np.array(y_test_list, dtype=np.int64)


In [ ]:
DESCRIPTOR_MODE = 'mean_var_grad'  # Options: 'mean', 'mean_var', 'mean_var_grad'
HIDDEN_CHANNELS = 8
NORMALIZE_DESCRIPTORS = True

class SNSM_Block(nn.Module):
    def __init__(self, channels, hidden_channels=8, mode='mean_var_grad'):
        super().__init__()
        self.mode = mode
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(channels)
        
        in_desc = 3 if mode == 'mean_var_grad' else (2 if mode == 'mean_var' else 1)
        self.spectral_conv1 = nn.Conv1d(in_channels=in_desc, out_channels=hidden_channels, kernel_size=3, padding=2, dilation=2, padding_mode='replicate')
        self.spectral_conv2 = nn.Conv1d(in_channels=hidden_channels, out_channels=2, kernel_size=3, padding=1, padding_mode='replicate')
        
        nn.init.kaiming_normal_(self.spectral_conv1.weight, mode='fan_out', nonlinearity='relu')
        nn.init.zeros_(self.spectral_conv2.weight)
        nn.init.zeros_(self.spectral_conv2.bias)
        nn.init.kaiming_normal_(self.conv1.weight, mode='fan_out', nonlinearity='relu')
        nn.init.kaiming_normal_(self.conv2.weight, mode='fan_out', nonlinearity='relu')

    def _normalize_descriptor(self, stat):
        if not NORMALIZE_DESCRIPTORS: return stat
        return (stat - stat.mean(dim=1, keepdim=True)) / (stat.std(dim=1, keepdim=True) + 1e-6)

    def forward(self, x):
        residual = x
        out = F.relu(self.bn1(self.conv1(x)))
        mean = torch.mean(out, dim=(2, 3))
        descriptors = [self._normalize_descriptor(mean)]
        
        if self.mode in ['mean_var', 'mean_var_grad']:
            var = torch.var(out, dim=(2, 3), unbiased=False)
            descriptors.append(self._normalize_descriptor(var))
            
        if self.mode == 'mean_var_grad':
            grad = torch.zeros_like(mean)
            grad[:, 1:-1] = (mean[:, 2:] - mean[:, :-2]) / 2.0
            grad[:, 0] = mean[:, 1] - mean[:, 0]
            grad[:, -1] = mean[:, -1] - mean[:, -2]
            descriptors.append(self._normalize_descriptor(grad))
            
        stats = torch.stack(descriptors, dim=1) 
        z = F.relu(self.spectral_conv1(stats))
        z = self.spectral_conv2(z)
        
        gamma = 1.0 + 0.5 * torch.tanh(z[:, 0, :]).unsqueeze(-1).unsqueeze(-1)
        beta  = z[:, 1, :].unsqueeze(-1).unsqueeze(-1)
        
        out = out * gamma + beta
        out = self.bn2(self.conv2(out))
        return F.relu(out + residual)

class SNSM_CNN(nn.Module):
    def __init__(self, num_classes, num_bands, hidden_channels=8, mode='mean_var_grad'):
        super().__init__()
        self.conv1 = nn.Conv2d(num_bands, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.block1 = SNSM_Block(64, hidden_channels=hidden_channels, mode=mode)
        self.block2 = SNSM_Block(64, hidden_channels=hidden_channels, mode=mode)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(p=0.4)
        self.fc = nn.Linear(64, num_classes)
        self.extracted_features = None

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.block1(x)
        x = self.block2(x)
        x = self.global_pool(x).flatten(1)
        self.extracted_features = x
        return self.fc(self.dropout(x))


In [ ]:
def calculate_fisher_ratio(features, labels, num_classes):
    global_mean = np.mean(features, axis=0)
    inter_variance, intra_variance = 0.0, 0.0
    for c in range(num_classes):
        class_feats = features[labels == c]
        if len(class_feats) == 0: continue
        class_mean = np.mean(class_feats, axis=0)
        inter_variance += len(class_feats) * np.sum((class_mean - global_mean)**2)
        intra_variance += np.sum((class_feats - class_mean)**2)
    return inter_variance / intra_variance if intra_variance != 0 else 0

def train_and_eval(model, train_loader, test_loader, device, num_classes, epochs=100):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    # Industry standard without val set: fixed epochs with step decay
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=40, gamma=0.5)
    
    for epoch in range(epochs):
        model.train()
        for data, target in train_loader:
            data, target = data.to(device, dtype=torch.float32), target.to(device, dtype=torch.long)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            
        scheduler.step()
    
    # Final Eval on Strictly Unseen Test Set
    model.eval()
    all_preds, all_targets, all_features = [], [], []
    with torch.no_grad():
        for data, target in test_loader:
            data = data.to(device, dtype=torch.float32)
            output = model(data)
            all_preds.extend(torch.argmax(output, dim=1).cpu().numpy())
            all_targets.extend(target.cpu().numpy())
            all_features.extend(model.extracted_features.cpu().numpy())
            
    all_targets, all_features = np.array(all_targets), np.array(all_features)
    oa, kappa = accuracy_score(all_targets, all_preds), cohen_kappa_score(all_targets, all_preds)
    cm = confusion_matrix(all_targets, all_preds)
    cm_sum = cm.sum(axis=1)
    aa = np.mean(np.divide(np.diag(cm), cm_sum, out=np.zeros_like(np.diag(cm), dtype=float), where=cm_sum!=0))
    fisher_ratio = calculate_fisher_ratio(all_features, all_targets, num_classes)
    return oa, aa, kappa, fisher_ratio


In [ ]:
datasets_to_run = ['IndianPines', 'PaviaU', 'Salinas', 'Botswana', 'KSC', 'WHU_Hi_HanChuan']
seeds = [42, 43, 44, 45, 46]
num_components = 30
window_size = 15

# Configurations to test:
# 1. Full Pipeline (With ASWD & Wavelet)
# 2. No ASWD (Only Wavelet)
# 3. No Wavelet (Only ASWD)
configs = [
    {"name": "Full_Pipeline", "use_denoise": True, "use_wavelet": True},
    {"name": "No_ASWD", "use_denoise": False, "use_wavelet": True},
    {"name": "No_Wavelet", "use_denoise": True, "use_wavelet": False}
]

for config in configs:
    csv_filename = f"results_{config['name']}.csv"
    with open(csv_filename, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(["Dataset", "OA (Mean ± Std)", "AA (Mean ± Std)", "Kappa (Mean ± Std)", "Fisher Ratio (Mean ± Std)"])
        
    print(f"\n{'='*60}\nSTARTING CONFIGURATION: {config['name']}\n{'='*60}")
    
    for dataset_name in datasets_to_run:
        print(f"\n--- Evaluating {dataset_name} ({config['name']}) ---")
        img, gt = load_dataset(dataset_name)
        
        if config['use_denoise']:
            img_denoised = apply_kalman_denoising(img)
        else:
            img_denoised = img
            
        img_pca = apply_pca(img_denoised, num_components=num_components)
        
        if config['use_wavelet']:
            img_features = apply_polar_wavelet_transform(img_pca)
        else:
            img_features = img_pca
            
        if config['use_denoise']: del img_denoised
        del img, img_pca
        gc.collect()
        
        num_classes = int(np.max(gt))
        
        oa_list, aa_list, kappa_list, fisher_list = [], [], [], []
        
        for seed in seeds:
            set_seed(seed)
            X_train, X_test, y_train, y_test = create_disjoint_patches(img_features, gt, window_size=window_size, train_ratio=0.05, seed=seed)
            
            MAX_TEST_SAMPLES = 50000
            if len(X_test) > MAX_TEST_SAMPLES:
                idx = np.random.choice(len(X_test), MAX_TEST_SAMPLES, replace=False)
                X_test, y_test = X_test[idx], y_test[idx]
                
            train_loader = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train)), batch_size=64, shuffle=True)
            test_loader = DataLoader(TensorDataset(torch.tensor(X_test), torch.tensor(y_test)), batch_size=64, shuffle=False)
            
            model = SNSM_CNN(num_classes=num_classes, num_bands=img_features.shape[2], 
                             hidden_channels=HIDDEN_CHANNELS, mode=DESCRIPTOR_MODE).to(device)
            
            oa, aa, kappa, fisher = train_and_eval(model, train_loader, test_loader, device, num_classes)
            oa_list.append(oa * 100); aa_list.append(aa * 100); kappa_list.append(kappa); fisher_list.append(fisher)
            
            del train_loader, test_loader, X_train, X_test, y_train, y_test, model
            gc.collect()
            
        with open(csv_filename, 'a', newline='') as f:
            csv.writer(f).writerow([dataset_name, 
                             f"{np.mean(oa_list):.2f} ± {np.std(oa_list):.2f}", 
                             f"{np.mean(aa_list):.2f} ± {np.std(aa_list):.2f}", 
                             f"{np.mean(kappa_list):.4f} ± {np.std(kappa_list):.4f}",
                             f"{np.mean(fisher_list):.2f} ± {np.std(fisher_list):.2f}"])


In [ ]:
# Architectural Ablations (from run_paper_experiments.py)
print("\n{'='*60}\nSTARTING ARCHITECTURAL ABLATIONS\n{'='*60}")
ablation_datasets = ['IndianPines', 'KSC', 'Botswana']
modes = ['mean', 'mean_var', 'mean_var_grad']
hiddens = [4, 8, 16]
norms = [True, False]

with open("architectural_ablations.csv", 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["Dataset", "Mode", "Hidden_Ch", "Normalize", "OA (Mean ± Std)", "AA (Mean ± Std)", "Kappa (Mean ± Std)", "Fisher (Mean ± Std)"])

for dataset_name in ablation_datasets:
    img, gt = load_dataset(dataset_name)
    img_denoised = apply_kalman_denoising(img)
    img_pca = apply_pca(img_denoised, num_components=num_components)
    img_features = apply_polar_wavelet_transform(img_pca)
    del img, img_denoised, img_pca
    gc.collect()
    num_classes = int(np.max(gt))
    
    for norm in norms:
        NORMALIZE_DESCRIPTORS = norm
        for mode in modes:
            for hidden in hiddens:
                print(f"\n[{dataset_name}] Evaluating: mode={mode}, hidden={hidden}, norm={norm}")
                oa_list, aa_list, kappa_list, fisher_list = [], [], [], []
                
                for seed in seeds:
                    set_seed(seed)
                    X_train, X_test, y_train, y_test = create_disjoint_patches(img_features, gt, window_size=window_size, train_ratio=0.05, seed=seed)
                    
                    train_loader = DataLoader(TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long)), batch_size=64, shuffle=True)
                    test_loader = DataLoader(TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long)), batch_size=64, shuffle=False)
                    
                    model = SNSM_CNN(num_classes=num_classes, num_bands=img_features.shape[2], 
                                     hidden_channels=hidden, mode=mode).to(device)
                    
                    oa, aa, kappa, fisher = train_and_eval(model, train_loader, test_loader, device, num_classes)
                    oa_list.append(oa * 100); aa_list.append(aa * 100); kappa_list.append(kappa); fisher_list.append(fisher)
                    
                    del train_loader, test_loader, X_train, X_test, y_train, y_test, model
                    gc.collect()
                    
                with open("architectural_ablations.csv", 'a', newline='') as f:
                    csv.writer(f).writerow([dataset_name, mode, hidden, norm,
                                     f"{np.mean(oa_list):.2f} ± {np.std(oa_list):.2f}", 
                                     f"{np.mean(aa_list):.2f} ± {np.std(aa_list):.2f}", 
                                     f"{np.mean(kappa_list):.4f} ± {np.std(kappa_list):.4f}",
                                     f"{np.mean(fisher_list):.2f} ± {np.std(fisher_list):.2f}"])
